In [ ]:
!pip install -q langchain-huggingface langchain-qdrant langchain-community sentence-transformers rank_bm25 nltk numpy pandas matplotlib

In [2]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
from nltk.tokenize import word_tokenize

In [4]:
QDRANT_URL = ""
QDRANT_API_KEY = ""

EMBED_MODEL = "abhinand/MedEmbed-base-v0.1"
COLLECTION_NAME = "ada_model_medembed_base_v0.1"
RERANKER_MODEL = "BAAI/bge-reranker-base"

QDRANT_TOP = 10  # semantic candidates from Qdrant
BM25_TOP = 10  # keyword candidates from BM25
MERGED_CAP = 20  # cap passed to cross-encoder (≤ QDRANT_TOP + BM25_TOP)
RERANK_TOP = 5  # final docs returned after reranking

Setting configs

In [5]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
EVAL_PATH = Path("/content/eval_queries_multi_chunk.json")
CHUNKS_PATH = Path("/content/drive/MyDrive/processed_chunks")

with open(EVAL_PATH) as f:
    eval_queries = json.load(f)

processed_chunks = []
for file_path in CHUNKS_PATH.glob("*.json"):
    with open(file_path, "r") as f:
        processed_chunks.extend(json.load(f))

In [7]:
print(f"Loaded {len(eval_queries)} queries")
print(f"Loaded {len(processed_chunks)} chunks for BM25 index")

Loaded 30 queries
Loaded 2632 chunks for BM25 index


Metrics

In [8]:
def precision_at_k(retrieved: list, relevant: list, k: int = 5) -> float:
    return len(set(retrieved[:k]) & set(relevant)) / k


def recall_at_k(retrieved: list, relevant: list, k: int = 10) -> float:
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)


def mrr(retrieved: list, relevant: list) -> float:
    rel_set = set(relevant)
    for i, cid in enumerate(retrieved, 1):
        if cid in rel_set:
            return 1.0 / i
    return 0.0


def ndcg_at_k(retrieved: list, scores: dict, k: int = 5) -> float:
    dcg = sum(
        scores.get(cid, 0) / np.log2(i + 1) for i, cid in enumerate(retrieved[:k], 1)
    )
    idcg = sum(
        s / np.log2(i + 1)
        for i, s in enumerate(sorted(scores.values(), reverse=True)[:k], 1)
    )
    return dcg / idcg if idcg > 0 else 0.0


def hit_at_k(retrieved: list, relevant: list, k: int = 5) -> float:
    return 1.0 if set(retrieved[:k]) & set(relevant) else 0.0

Building QDrant retriever and BM25 index

In [ ]:
# Load MedEmbed + connect to existing Qdrant collection
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = QdrantVectorStore.from_existing_collection(
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)
qdrant_retriever = vector_store.as_retriever(search_kwargs={"k": QDRANT_TOP})

In [10]:
# Build BM25 index from processed_chunks.json using word_tokenize
bm25_docs = [
    Document(
        page_content=chunk.get("page_content", ""),
        metadata={"chunk_id": chunk["metadata"]["chunk_id"]},
    )
    for i, chunk in enumerate(processed_chunks)
]
bm25_retriever = BM25Retriever.from_documents(
    bm25_docs,
    k=BM25_TOP,
    preprocess_func=word_tokenize,
)

print(f"BM25 index built over {len(bm25_docs)} chunks (top-{BM25_TOP})")

BM25 index built over 2632 chunks (top-10)


Reciprocal Rank Fusion via Langchain's EnsembleRetriever

In [11]:
rrf_retriever = EnsembleRetriever(
    retrievers=[qdrant_retriever, bm25_retriever],
    weights=[0.7, 0.3],
)

In [ ]:
cross_encoder = CrossEncoder(RERANKER_MODEL)
print(f"✓ Reranker loaded: {RERANKER_MODEL}")

Eval Loop

In [13]:
def evaluate_hybrid() -> dict:
    """BM25 + MedEmbed via EnsembleRetriever (RRF 0.7/0.3) → bge-reranker-base → top-5."""
    p5_list, r10_list, mrr_list, ndcg5_list, hit5_list, lat_list = (
        [],
        [],
        [],
        [],
        [],
        [],
    )

    for i, item in enumerate(eval_queries, 1):
        query = item["query"]
        relevant_ids = item["ground_truth"]["relevant_chunk_ids"]
        rel_scores = item["ground_truth"]["relevance_scores"]

        t0 = time.time()

        # EnsembleRetriever fuses Qdrant + BM25 via RRF
        rrf_docs = rrf_retriever.invoke(query)
        candidates = rrf_docs[:MERGED_CAP]

        # Cross-encoder reranking: score every (query, passage) pair
        passages = [doc.page_content for doc in candidates]
        ce_inputs = [[query, p] for p in passages]
        ce_scores = cross_encoder.predict(ce_inputs)

        ranked_docs = sorted(
            zip(ce_scores, candidates), key=lambda x: x[0], reverse=True
        )
        top_docs = [doc for _, doc in ranked_docs[:RERANK_TOP]]
        lat = time.time() - t0

        # Extract chunk IDs for metric
        retrieved_ids = [doc.metadata.get("chunk_id") for doc in top_docs]
        all_reranked = [doc.metadata.get("chunk_id") for _, doc in ranked_docs]

        p5 = precision_at_k(retrieved_ids, relevant_ids, k=5)
        r10 = recall_at_k(all_reranked, relevant_ids, k=10)
        mrr_s = mrr(retrieved_ids, relevant_ids)
        ndcg5 = ndcg_at_k(retrieved_ids, rel_scores, k=5)
        hit5 = hit_at_k(retrieved_ids, relevant_ids, k=5)

        p5_list.append(p5)
        r10_list.append(r10)
        mrr_list.append(mrr_s)
        ndcg5_list.append(ndcg5)
        hit5_list.append(hit5)
        lat_list.append(lat)

        print(
            f"  [{i:02d}] P@5={p5:.2f} NDCG@5={ndcg5:.2f} Hit@5={int(hit5)} "
            f"candidates={len(candidates):02d} | {lat * 1000:.0f}ms | {query[:50]}…"
        )

    return {
        "Pipeline": f"MedEmbed + BM25 RRF [0.7, 0.3]\n + bge-reranker-base",
        "Precision@5": round(float(np.mean(p5_list)), 3),
        "Recall@10": round(float(np.mean(r10_list)), 3),
        "MRR": round(float(np.mean(mrr_list)), 3),
        "NDCG@5": round(float(np.mean(ndcg5_list)), 3),
        "Hit@5": round(float(np.mean(hit5_list)), 3),
        "Latency_ms": round(float(np.mean(lat_list)) * 1000, 1),
    }


rrf_w_reranker = evaluate_hybrid()
print(f"\n  Precision@5 : {rrf_w_reranker['Precision@5']}")
print(f"  Recall@10   : {rrf_w_reranker['Recall@10']}")
print(f"  MRR         : {rrf_w_reranker['MRR']}")
print(f"  NDCG@5      : {rrf_w_reranker['NDCG@5']}")
print(f"  Hit@5       : {rrf_w_reranker['Hit@5']}")
print(f"  Latency     : {rrf_w_reranker['Latency_ms']} ms")

  [01] P@5=0.60 NDCG@5=0.98 Hit@5=1 candidates=14 | 1280ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=17 | 597ms | What patient education should be provided to an in…
  [03] P@5=0.40 NDCG@5=0.29 Hit@5=1 candidates=15 | 566ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.80 NDCG@5=0.90 Hit@5=1 candidates=18 | 674ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.80 NDCG@5=0.93 Hit@5=1 candidates=18 | 659ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.00 NDCG@5=0.00 Hit@5=0 candidates=16 | 604ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.40 NDCG@5=0.60 Hit@5=1 candidates=18 | 598ms | A 45-year-old patient with type 2 diabetes and obe…
  [08] P@5=0.60 NDCG@5=0.57 Hit@5=1 candidates=15 | 748ms | A 65-year-old patient with diabetes is having trou…
  [09] P@5=0.00 NDCG@5=0.00 Hit@5=0 candidates=19 | 833ms | A 40-year-old patient with type 1 diabetes 

In [14]:
rrf_w_reranker

{'Pipeline': 'MedEmbed + BM25 RRF [0.7, 0.3]\n + bge-reranker-base',
 'Precision@5': 0.393,
 'Recall@10': 0.59,
 'MRR': 0.722,
 'NDCG@5': 0.47,
 'Hit@5': 0.867,
 'Latency_ms': 717.1}